# Diabetic Readmission Risk: Data Assessment and Cleaning

**Source:** UCI Diabetes 130-US Hospitals dataset (Strack et al., 2014) — 101,766 
inpatient diabetic encounters across 130 hospitals, 1999–2008.

**Purpose:** Prepare this data for a clinical decision-support dashboard examining 
how HbA1c testing and patient utilization history relate to 30-day readmission risk.

**Key findings from this assessment:**
- HbA1c was tested in only ~18% of encounters, despite being the primary lever 
  this analysis will examine
- Two fields (`weight`, `max_glu_serum`) are too sparse to support any reliable 
  finding and are excluded rather than imputed
- Three ID fields (admission type, discharge disposition, admission source) are 
  coded as integers but are categorical lookups — treating them as numbers would 
  silently corrupt any downstream analysis
- 24 individual medication columns represent one underlying variable (drug dosage 
  change) and will be restructured for analysis

The sections below document the assessment and the specific cleaning decisions 
made, with rationale for each.

## Data Loading

Two fields require explicit type handling on load — not because the source data 
is wrong, but because pandas' default type inference would silently corrupt them 
(e.g., stripping meaningful leading characters). Loading correctly the first time 
avoids re-deriving from source later.

In [1]:
import pandas as pd
print(pd.__version__)

3.0.5


In [2]:
diabetic = pd.read_csv(
    '../data/raw/diabetic_data.csv',
    keep_default_na=False,
    na_values=['?'],
    dtype={'payer_code': str},
    low_memory=False
)
print(f"Loaded {diabetic.shape[0]:,} encounters, {diabetic.shape[1]} fields")

Loaded 101,766 encounters, 50 fields


## Data Quality Issues Found

The table below summarizes what matters for the dashboard — not every column, 
but every issue that affects a downstream decision.

In [3]:
missing = diabetic.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(diabetic) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary[missing_summary['missing_count'] > 0]

,missing_count,missing_pct
weight,98569,96.9
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0


**Completeness**
- `weight`: 96.9% missing — excluded from analysis. There's not enough signal 
  in the remaining 3.1% to support any reliable claim, and imputing it would 
  manufacture data that doesn't exist.
- `medical_specialty` (49.1%) and `payer_code` (39.6%): incomplete but 
  informative — kept, with missingness itself flagged as a category, since who 
  doesn't get recorded can be as meaningful as who does.

**Validity**
- `admission_type_id`, `discharge_disposition_id`, `admission_source_id` are 
  stored as integers (1, 2, 3...) but represent categories (Emergency, Elective, 
  Discharged to home...), not quantities. Left as numbers, they'd be silently 
  averaged or ranked in later analysis — a real risk, not a theoretical one. 
  Mapped to their descriptions via `IDS_mapping.csv` (split into three lookup 
  tables — see Cleaning Decisions below).
- A subset of codes in each of the three lookup tables are themselves labeled 
  `"NULL"` or `"Not Mapped"` by the source (e.g., `admission_type_id = 6`, 
  `discharge_disposition_id = 18`, `admission_source_id = 17`) — meaning even 
  UCI/CMS never documented what these codes represent. These are real, valid 
  categories (the code exists in the data) and are preserved as-is rather than 
  treated as missing.

**Consistency**
- `max_glu_serum` and `A1Cresult` both use the string `"None"` to mean "not 
  tested" — a real category, not a null value. Handled explicitly rather than 
  coerced to NaN, to avoid losing this distinction.

**Tidiness**
- 24 separate columns (`metformin`, `insulin`, `glipizide`, etc.) all represent 
  one underlying variable — whether and how a diabetes medication's dosage 
  changed during the encounter. Kept wide for this notebook's inspection, but 
  will be reshaped to long format for the medication-change analysis.

### Check for duplicate patient encounters
Reason: the source paper used only one encounter per patient to keep 
observations independent for their model. Before deciding whether to follow 
that approach, we need to know whether duplicates actually exist in this data.

In [4]:
dup_patients = diabetic['patient_nbr'].duplicated().sum()
print(f"Encounters: {len(diabetic):,}")
print(f"Unique patients: {diabetic['patient_nbr'].nunique():,}")
print(f"Patients with more than one encounter: {dup_patients:,}")

Encounters: 101,766
Unique patients: 71,518
Patients with more than one encounter: 30,248


### Check whether repeat encounters represent meaningfully different treatment
Reason: before deciding whether to keep or filter repeat patients, verify 
whether their encounters differ in treatment and outcome — if they do, 
filtering to one encounter per patient would discard real information.

In [5]:
# Isolate patients with more than one encounter
repeat_ids = diabetic['patient_nbr'].value_counts()
repeat_ids = repeat_ids[repeat_ids > 1].index
repeats = diabetic[diabetic['patient_nbr'].isin(repeat_ids)]

print(f"Repeat-patient encounters: {len(repeats):,}")

# For each repeat patient, how much does their treatment vary across visits?
variation = repeats.groupby('patient_nbr').agg(
    n_encounters=('encounter_id', 'count'),
    distinct_primary_diag=('diag_1', 'nunique'),
    distinct_admission_type=('admission_type_id', 'nunique'),
    distinct_num_medications=('num_medications', 'nunique'),
    distinct_readmitted_outcome=('readmitted', 'nunique'),
)

print(variation.describe())

Repeat-patient encounters: 47,021
       n_encounters  distinct_primary_diag  distinct_admission_type  \
count  16773.000000           16773.000000             16773.000000   
mean       2.803374               2.356585                 1.504561   
std        1.607310               1.120199                 0.575293   
min        2.000000               1.000000                 1.000000   
25%        2.000000               2.000000                 1.000000   
50%        2.000000               2.000000                 1.000000   
75%        3.000000               3.000000                 2.000000   
max       40.000000              15.000000                 4.000000   

       distinct_num_medications  distinct_readmitted_outcome  
count              16773.000000                 16773.000000  
mean                   2.610326                     1.970309  
std                    1.256111                     0.547048  
min                    1.000000                     1.000000  
25%        

**Finding:** repeat visits differ substantially from each other. A typical 
repeat patient had ~2 distinct primary diagnoses and 2 of 3 possible 
readmission outcomes across their visits — evidence these are genuinely 
different clinical episodes, not duplicated records.

**Decision:** retain all encounters rather than filtering to one per patient 
(as the source paper did). Filtering would discard real signal, particularly 
variation in readmission outcome — the core variable this project studies. 
Repeat patients are flagged instead (see below), so the dashboard can still 
segment or filter by this if needed.

In [6]:
patient_encounter_counts = diabetic['patient_nbr'].value_counts()
diabetic['is_repeat_patient'] = diabetic['patient_nbr'].map(
    lambda x: patient_encounter_counts[x] > 1
)

assert diabetic['is_repeat_patient'].sum() == 47021
print(diabetic['is_repeat_patient'].value_counts())

is_repeat_patient
False    54745
True     47021
Name: count, dtype: int64


**Result:** 47,021 encounters (46.2% of the dataset) belong to patients with 
more than one visit; 54,745 encounters are single, unrepeated visits. All 
encounters are retained — `is_repeat_patient` allows the dashboard to filter 
or compare outcomes between first-time and returning patients without 
discarding either group.

## Cleaning Decisions
Each decision below states what was done and why, followed by a check confirming 
the result is correct — not just that code ran without error.

### Drop `weight`
96.9% missing — dropped, not imputed.

In [7]:
diabetic = diabetic.drop(columns=['weight'])
assert 'weight' not in diabetic.columns
print("weight column dropped and confirmed removed")

weight column dropped and confirmed removed


### Confirm "not tested" markers preserved correctly
Verifying `None` survived the load as a real category, not as missing data.

In [8]:
print(diabetic['A1Cresult'].value_counts(dropna=False))
print(diabetic['max_glu_serum'].value_counts(dropna=False))

A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64
max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64


**Result:** confirmed — `None` correctly labeled in both fields (84,748 
`A1Cresult`, 96,420 `max_glu_serum`), not converted to missing.

### Map coded ID fields to their real-world meaning
Reason: these three fields are categorical lookups, not numbers. Mapping now 
prevents accidental numeric misuse (averaging, ranking) later and makes the 
dashboard readable without a separate codebook.

Note: the lookup files use the literal text `"NULL"` as one specific category 
(an undocumented-but-real code), not as a missing-value marker. Loaded with 
`keep_default_na=False` to prevent pandas from silently converting that text 
into an actual null — which was initially causing valid rows to fail validation 
after merging.

In [9]:
admission_type = pd.read_csv('../data/raw/admission_type.csv', keep_default_na=False, na_values=[''])
print(admission_type)

   admission_type_id    description
0                  1      Emergency
1                  2         Urgent
2                  3       Elective
3                  4        Newborn
4                  5  Not Available
5                  6           NULL
6                  7  Trauma Center
7                  8     Not Mapped


In [10]:
diabetic = diabetic.merge(
    admission_type, on='admission_type_id', how='left'
).rename(columns={'description': 'admission_type_desc'})

assert diabetic['admission_type_desc'].isnull().sum() == 0, "Unmapped admission_type_id values found"
diabetic[['admission_type_id', 'admission_type_desc']].drop_duplicates()

,admission_type_id,admission_type_desc
0,6,NULL
1,1,Emergency
5,2,Urgent
6,3,Elective
2043,4,Newborn
3089,5,Not Available
7789,8,Not Mapped
45829,7,Trauma Center


In [11]:
discharge_disposition = pd.read_csv('../data/raw/discharge_disposition.csv', keep_default_na=False, na_values=[''])
diabetic = diabetic.merge(
    discharge_disposition, on='discharge_disposition_id', how='left'
).rename(columns={'description': 'discharge_disposition_desc'})

assert diabetic['discharge_disposition_desc'].isnull().sum() == 0, "Unmapped discharge_disposition_id values found"
diabetic[['discharge_disposition_id', 'discharge_disposition_desc']].drop_duplicates()

,discharge_disposition_id,discharge_disposition_desc
0,25,Not Mapped
1,1,Discharged to home
9,3,Discharged/transferred to SNF
13,6,Discharged/transferred to home with home healt...
29,2,Discharged/transferred to another short term h...
31,5,Discharged/transferred to another type of inpa...
34,11,Expired
82,7,Left AMA
487,10,Neonate discharged to another hospital for neo...
972,4,Discharged/transferred to ICF


In [12]:
admission_source = pd.read_csv('../data/raw/admission_source.csv', keep_default_na=False, na_values=[''])
diabetic = diabetic.merge(
    admission_source, on='admission_source_id', how='left'
).rename(columns={'description': 'admission_source_desc'})

assert diabetic['admission_source_desc'].isnull().sum() == 0, "Unmapped admission_source_id values found"
diabetic[['admission_source_id', 'admission_source_desc']].drop_duplicates()

,admission_source_id,admission_source_desc
0,1,Physician Referral
1,7,Emergency Room
5,2,Clinic Referral
8,4,Transfer from a hospital
335,5,Transfer from a Skilled Nursing Facility (SNF)
396,6,Transfer from another health care facility
469,20,Not Mapped
1014,3,HMO Referral
1039,17,NULL
1595,8,Court/Law Enforcement


In [13]:
# Verification: confirm no admission_type_id, discharge_disposition_id, or
# admission_source_id failed to find a matching description after merge
assert diabetic['admission_type_desc'].isnull().sum() == 0
assert diabetic['discharge_disposition_desc'].isnull().sum() == 0
assert diabetic['admission_source_desc'].isnull().sum() == 0
print("All ID fields successfully mapped — no unmatched codes")

All ID fields successfully mapped — no unmatched codes


**Result:** all three ID fields mapped successfully across 101,766 
encounters, zero unmatched codes.

### Fill `payer_code` and `medical_specialty` with "Not Recorded"
Kept rather than dropped — who lacks a recorded payer or specialty is itself 
informative. Missing values relabeled explicitly rather than left as NaN, so 
they remain visible in dashboard filters instead of silently dropping out.

In [14]:
diabetic['payer_code'] = diabetic['payer_code'].fillna('Not Recorded')
diabetic['medical_specialty'] = diabetic['medical_specialty'].fillna('Not Recorded')

assert diabetic['payer_code'].isnull().sum() == 0
assert diabetic['medical_specialty'].isnull().sum() == 0

print(diabetic['payer_code'].value_counts().head())
print(diabetic['medical_specialty'].value_counts().head())

payer_code
Not Recorded    40256
MC              32439
HM               6274
SP               5007
BC               4655
Name: count, dtype: int64
medical_specialty
Not Recorded              49949
InternalMedicine          14635
Emergency/Trauma           7565
Family/GeneralPractice     7440
Cardiology                 5352
Name: count, dtype: int64


**Result:** 40,256 `payer_code` and 49,949 `medical_specialty` values 
relabeled; both fields now complete.

### Add a binary readmission flag, without discarding the original detail
Reason: the primary analysis question is 30-day readmission risk, which needs 
a binary target. The original three-category column (`<30`, `>30`, `NO`) is 
preserved rather than overwritten, ...so future analyses remain possible without re-deriving from raw data.

In [15]:
diabetic['readmitted_30d'] = (diabetic['readmitted'] == '<30').astype(int)

assert diabetic['readmitted_30d'].isin([0, 1]).all()
print(diabetic[['readmitted', 'readmitted_30d']].value_counts())

readmitted  readmitted_30d
NO          0                 54864
>30         0                 35545
<30         1                 11357
Name: count, dtype: int64


### Inspect primary and secondary diagnosis codes
Reason: these are raw ICD-9 codes (numeric ranges and V/E-prefixed codes). 
Before any grouping decision, confirm their structure and cardinality.

In [16]:
print("Unique diag_1 codes:", diabetic['diag_1'].nunique())
print("Unique diag_2 codes:", diabetic['diag_2'].nunique())
print("Unique diag_3 codes:", diabetic['diag_3'].nunique())
print(diabetic['diag_1'].value_counts().head(10))

Unique diag_1 codes: 716
Unique diag_2 codes: 748
Unique diag_3 codes: 789
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64


### Group `diag_1` into broad diagnostic categories
Reason: 716 distinct raw ICD-9 codes are too granular for dashboard filtering 
or visualization. Grouped into the same 9 categories used by Strack et al. 
(2014, Table 2), based on `diag_1` only — the primary diagnosis, which 
represents the main reason for admission. `diag_2` and `diag_3` (secondary 
diagnoses) are left as raw codes for now, since grouping them is only useful 
for a comorbidity-focused analysis this project hasn't yet scoped.

Category ranges (ICD-9):
- Circulatory: 390–459, 785
- Respiratory: 460–519, 786
- Digestive: 520–579, 787
- Diabetes: 250.xx
- Injury: 800–999
- Musculoskeletal: 710–739
- Genitourinary: 580–629, 788
- Neoplasms: 140–239
- Other: everything

In [17]:
def group_diagnosis(code):
    """
    Map a raw ICD-9 diag_1 code to one of the paper's 9 broad categories.
    Non-numeric codes (V-codes, E-codes) and anything outside the defined
    ranges fall into 'Other'.
    """
    if pd.isna(code):
        return 'Missing'
    
    try:
        code_num = float(code)
    except ValueError:
        # V-codes and E-codes (e.g. 'V27', 'E812') aren't convertible to float
        return 'Other'
    
    if 390 <= code_num <= 459 or code_num == 785:
        return 'Circulatory'
    elif 460 <= code_num <= 519 or code_num == 786:
        return 'Respiratory'
    elif 520 <= code_num <= 579 or code_num == 787:
        return 'Digestive'
    elif 250 <= code_num < 251:
        return 'Diabetes'
    elif 800 <= code_num <= 999:
        return 'Injury'
    elif 710 <= code_num <= 739:
        return 'Musculoskeletal'
    elif 580 <= code_num <= 629 or code_num == 788:
        return 'Genitourinary'
    elif 140 <= code_num <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

diabetic['diag_1_group'] = diabetic['diag_1'].apply(group_diagnosis)

assert diabetic['diag_1_group'].isnull().sum() == 0
print(diabetic['diag_1_group'].value_counts())

diag_1_group
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Missing               21
Name: count, dtype: int64


In [18]:
diabetic[diabetic['diag_1_group'] == 'Other']['diag_1'].value_counts().head(15)

diag_1
682    2042
780    2019
276    1889
38     1688
V57    1207
296     896
789     561
8       515
295     447
278     379
285     365
280     319
648     285
707     257
V58     228
Name: count, dtype: int64

**Verification:** top codes landing in "Other" (skin conditions, ill-defined 
symptoms, non-diabetes endocrine disorders, V-codes) all fall in ranges the 
source paper also folds into its own "Other" category. 17.9% here vs. their 
17.3% — consistent, no grouping errors found.

### Medication columns: deferred restructuring
Reason: 24 individual drug columns (plus 5 combination-drug columns) each 
represent whether/how a specific diabetes medication's dosage changed during 
the encounter — structurally, these are all one variable ("drug dosage 
change") spread across many columns, a tidiness violation. However, the 
source paper's own analysis never uses these at the individual-drug level — 
it relies entirely on two existing aggregate columns, `change` and 
`diabetesMed`, verified below. Reshaping to long format would be necessary 
only for a drug-specific analysis (e.g. "did insulin changes affect 
readmission differently than oral medication changes") — a question this 
project hasn't yet scoped. Reshaping now, before that need is confirmed, 
risks building structure that doesn't match the eventual dashboard's actual 
requirements. Deferred as a documented future-work item; base table remains 
wide, one row per encounter.

In [19]:
print(diabetic['change'].value_counts(dropna=False))
print(diabetic['diabetesMed'].value_counts(dropna=False))

assert diabetic['change'].isnull().sum() == 0
assert diabetic['diabetesMed'].isnull().sum() == 0
print("Both aggregate medication columns confirmed clean, no missing values")

change
No    54755
Ch    47011
Name: count, dtype: int64
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64
Both aggregate medication columns confirmed clean, no missing values


In [20]:
print(diabetic.shape)
print(diabetic.columns.tolist())

(101766, 55)
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'is_repeat_patient', 'admission_type_desc', 'discharge_disposition_desc', 'admission_source_desc', 'readmitted_30d', 'diag_1_group']


## Summary

| Decision | Result |
|---|---|
| Dropped `weight` | 96.9% missing |
| `max_glu_serum`/`A1Cresult` "None" preserved | confirmed, not coerced to missing |
| Three ID fields mapped | 0 unmatched codes |
| `payer_code`/`medical_specialty` filled | "Not Recorded" category, both complete |
| Repeat patients flagged, not filtered | 47,021 of 101,766 encounters |
| `readmitted_30d` added | additive, original 3-category column retained |
| `diag_1` grouped into 9 categories | `diag_2`/`diag_3` left raw — deferred, unscoped |
| Medication columns | deferred — aggregate `change`/`diabetesMed` used instead |

**Final shape:** 101,766 rows × 55 columns. Ready for schema design.

In [21]:
diabetic.to_csv('../data/processed/diabetic_data_cleaned.csv', index=False)
print(f"Saved {diabetic.shape[0]:,} rows × {diabetic.shape[1]} columns to data/processed/")

Saved 101,766 rows × 55 columns to data/processed/
